# 04 — Добор недостающих языков (IT, PT, JA, PL)

В Yandex-датасете покрыто 4 языка (RU/ES/EN/FR). ТЗ требует 8 (+ IT/PT/JA/PL). Добираем по 1 треку:
1. yt-dlp скачивает audio из YouTube
2. demucs htdemucs v4 → vocal m4a (та же утилита, что в проде Unbake)
3. LRCLib даёт лирику
4. baseline + alignment как для основного датасета

**Disclaimer:** YouTube как источник — для proof-of-architecture теста. В проде — лицензированные мастера/lossless источники.

Запускать на **Colab T4** ПОСЛЕ 03_full_bench (использует ту же машину).

Время: ~5-8 минут (4 трека × ~2 мин demucs/baseline/align).

In [ ]:
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/RezPoint/unbake-test.git 2>/dev/null; true
%cd unbake-test
!pip install -q faster-whisper transformers torchaudio librosa jiwer pydantic requests yt-dlp demucs

In [ ]:
# Тест-треки. youtube_id = последняя часть URL (?v=XXXX).
EXTRA = [
    {'lang': 'it', 'artist': 'Måneskin',     'track': 'I WANNA BE YOUR SLAVE', 'youtube_id': 'EE_zwroI2EU'},
    {'lang': 'pt', 'artist': 'Anitta',       'track': 'Envolver',              'youtube_id': 'X0DPpv0pPp8'},
    {'lang': 'ja', 'artist': 'YOASOBI',      'track': 'Idol',                  'youtube_id': 'ZRtdQ81jPUQ'},
    {'lang': 'pl', 'artist': 'sanah',        'track': 'Szampan',               'youtube_id': '6Vt6gYxqaXk'},
]

In [ ]:
# 1. Скачиваем audio через yt-dlp (m4a, 256kbps если есть)
import os, subprocess
from pathlib import Path
DOWNLOAD_DIR = Path('data/_yt_raw'); DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
for t in EXTRA:
    out = DOWNLOAD_DIR / f"{t['lang']}_{t['artist']}_{t['track']}.m4a".replace(' ', '_')
    if out.exists():
        print(f'skip {out.name}')
        continue
    cmd = ['yt-dlp', '-q', '-f', 'ba[ext=m4a]/ba', '--no-playlist',
           '-o', str(out), f"https://www.youtube.com/watch?v={t['youtube_id']}"]
    print('downloading', out.name)
    subprocess.run(cmd, check=True)
    t['raw_path'] = str(out)
for t in EXTRA:
    if 'raw_path' not in t:
        t['raw_path'] = str(DOWNLOAD_DIR / f"{t['lang']}_{t['artist']}_{t['track']}.m4a".replace(' ', '_'))

In [ ]:
# 2. demucs htdemucs v4 → vocal m4a
OUT_RAW = Path('data/raw'); OUT_RAW.mkdir(parents=True, exist_ok=True)
DEMUCS_OUT = Path('data/_demucs'); DEMUCS_OUT.mkdir(parents=True, exist_ok=True)
for t in EXTRA:
    lang_dir = OUT_RAW / t['lang']; lang_dir.mkdir(parents=True, exist_ok=True)
    final = lang_dir / f"{t['artist']} - {t['track']}.m4a"
    if final.exists():
        print('skip', final.name); t['vocal_path'] = str(final); continue
    cmd = ['python', '-m', 'demucs', '-n', 'htdemucs', '--two-stems=vocals',
           '--mp3' if False else '-o', str(DEMUCS_OUT), t['raw_path']]
    # demucs writes WAV by default. Convert to m4a 256k via ffmpeg.
    print('demucs:', t['raw_path'])
    subprocess.run(cmd, check=True)
    # demucs creates DEMUCS_OUT/htdemucs/<basename>/vocals.wav
    base = Path(t['raw_path']).stem
    src = DEMUCS_OUT / 'htdemucs' / base / 'vocals.wav'
    subprocess.run(['ffmpeg', '-y', '-i', str(src), '-c:a', 'aac', '-b:a', '256k', str(final)], check=True)
    t['vocal_path'] = str(final)
    print('→', final.name)

In [ ]:
# 3. Лирики через LRCLib
from src.lyrics import search
import time
for t in EXTRA:
    out = Path('data/lyrics') / t['lang']; out.mkdir(parents=True, exist_ok=True)
    lyr_path = out / f"{t['artist']} - {t['track']}.txt"
    if lyr_path.exists():
        print(f'skip {lyr_path.name}'); continue
    res = search(t['track'], t['artist'])
    if res is None:
        print(f'MISS {t["artist"]} - {t["track"]}'); continue
    lyr_path.write_text(res.plain_lyrics, encoding='utf-8')
    print(f'ok  {lyr_path.name} ({len(res.plain_lyrics.split())} words)')
    time.sleep(0.5)

In [ ]:
# 4. Запускаем batch_runner — он подхватит новые файлы в data/raw/<lang>/, кэш старых не тронет
!python -m src.eval.batch_runner

In [ ]:
import pandas as pd
df = pd.read_csv('docs/results.csv')
print(df.to_string(index=False))

In [ ]:
# Архив для скачивания
!cd data/transcripts && tar czf /content/transcripts.tar.gz *.json
!cp docs/results.csv /content/results.csv
print('downloaded:')
print('  /content/transcripts.tar.gz')
print('  /content/results.csv')